# Pipeline D — late fusion (baseline) — results

**Presentation notebook.** Pipeline D has *no training*: it merges the object lists of the trained **LiDAR-only** and **camera-only** detectors and scores the union. Needs their two checkpoints (train `MODEL='lidar'` and `'camera'` first).

## 0. Setup

In [ ]:
# Repo-root bootstrap — lets this notebook run from notebooks/ or the repo root.
import os, sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "globals.py").exists())
os.chdir(_ROOT)
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
print("repo root:", _ROOT)

In [ ]:
import os
from pathlib import Path
import data  # noqa: F401
_REPO = Path(data.__file__).resolve().parent
os.environ.setdefault("PY123D_DATA_ROOT", str(_REPO / "data"))
os.environ.setdefault("KITTI360_DATA_ROOT", str(_REPO / "KITTI-360"))
import torch
import globals as G
import utils
from data import Py123dDataset, stereo_cache_root
from evaluation import (CenterPointDecoder, evaluate_late_fusion, load_report,
                        merge_detections, print_ap_report, save_report, _det_to_numpy)
from network import CameraOnlyDetector, LidarOnlyDetector, StereoBEVConfig, lidar_points
import igev_matcher  # noqa: F401
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = "results/pipeline_d.json"
LIDAR_CKPT = "checkpoints/lidar.pt"
CAMERA_CKPT = "checkpoints/camera_yolo26_igev.pt"
FIGDIR = Path("docs/img"); FIGDIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| classes:", G.CLASSES)

## 1. Load the two single-sensor detectors

In [ ]:
val_ds = Py123dDataset(split_names=["kitti360_val"]); val_frames = list(val_ds)
CACHE = stereo_cache_root(val_ds.data_root, matcher="igev")
lm = LidarOnlyDetector(); lm.load_state_dict(torch.load(LIDAR_CKPT, weights_only=True)["model"]); lm.to(DEVICE).eval()
cm = CameraOnlyDetector(stereo_cache_root=CACHE, stereo_cfg=StereoBEVConfig(img_backbone="yolo26", yolo_freeze=True))
cm.load_state_dict(torch.load(CAMERA_CKPT, weights_only=True)["model"]); cm.to(DEVICE).eval()
print("loaded lidar + camera checkpoints")

## 2. Detection AP

In [ ]:
if Path(RESULTS).exists():
    report = load_report(RESULTS)
else:
    report = evaluate_late_fusion(lm, cm, val_frames, device=DEVICE, merge_radius_m=1.0)
    save_report(report, RESULTS)
print_ap_report(report)
print(f"\nHEADLINE  mAP {report['mAP']:.3f}  |  macro F1 {report['f1']:.3f} "
      f"@{report['op_threshold_m']:g} m  |  mean centre err {report['mean_error_m']:.3f} m")

## 3. Diagnostics

In [ ]:
utils.visualize_evaluation(report, save_path=str(FIGDIR / "pipeline_d_diagnostics.png"))
utils.visualize_evaluation(report)

## 4. Qualitative — merged detections vs GT

In [ ]:
decoder = CenterPointDecoder(score_threshold=0.2)
for i, frame in enumerate(val_frames[20:400:90]):
    sample = frame.to_stereo_sample(load_images=False, point_mask=False)
    with torch.no_grad():
        lo = lm(lidar_points(sample), device=DEVICE); co = cm(sample, device=DEVICE)
    dl = _det_to_numpy(decoder(lo["heatmap"].cpu(), lo["offset"].cpu())[0])
    dc = _det_to_numpy(decoder(co["heatmap"].cpu(), co["offset"].cpu())[0])
    merged = merge_detections([dl, dc], radius_m=1.0)
    det = {"boxes_2d": torch.as_tensor(merged["xy"], dtype=torch.float32),
           "scores": torch.as_tensor(merged["scores"], dtype=torch.float32),
           "classes": torch.as_tensor(merged["classes"], dtype=torch.long)}
    utils.visualize_detections(sample, det)                                   # inline
    utils.visualize_detections(sample, det,                                   # + save for slides
        save_path=str(FIGDIR / f"pipeline_d_det_{i}.png"))
print("saved qualitative BEV frames ->", FIGDIR)

---
*Cross-model comparison: `confronto.ipynb`.*